[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitm-da/da2402/blob/master/data%20collection/beautifulsoup_step_by_step.ipynb)

# BeautifulSoup, step by step

Companion notebook to **Lecture 2 — Data collection · HTML you can parse**.

Every idea here is shown on an HTML snippet small enough to read whole. Run the cells in order —
then *change the snippets and re-run*. Breaking a selector on purpose is the fastest way to learn
what the parser actually does. At the end we scale up from toy snippets to the real saved listings
page from the lecture.

## Setup

BeautifulSoup ships with Colab. On your own machine, uncomment the install line once.

In [1]:
# !pip install beautifulsoup4 pandas requests

In [2]:
from bs4 import BeautifulSoup
import bs4
bs4.__version__

'4.14.2'

## 1 · Parse: text in, tree out

To Python, HTML is just a string. `BeautifulSoup(text, "html.parser")` turns that string into a
**tree of objects** — the same tree your browser builds (the DOM from the lecture).

In [3]:
html = """
<html>
  <head><title>My first page</title></head>
  <body>
    <h1>Hello, HTML</h1>
    <p>A paragraph with a <a href="https://www.example.com">link</a> inside it.</p>
  </body>
</html>"""

soup = BeautifulSoup(html, "html.parser")
type(soup)

bs4.BeautifulSoup

In [4]:
# prettify() re-indents the tree - handy for seeing the structure the parser understood
print(soup.prettify())

<html>
 <head>
  <title>
   My first page
  </title>
 </head>
 <body>
  <h1>
   Hello, HTML
  </h1>
  <p>
   A paragraph with a
   <a href="https://www.example.com">
    link
   </a>
   inside it.
  </p>
 </body>
</html>



## 2 · Tags are objects

Every element becomes a `Tag`. A tag knows its **name**, its **attributes**, its **text**, and
its place in the tree (parent, children, siblings). `soup.h1` is shorthand for
"the *first* `<h1>` anywhere".

In [5]:
soup.title

<title>My first page</title>

In [6]:
soup.title.name

'title'

In [7]:
soup.title.string

'My first page'

In [8]:
soup.a['href']

'https://www.example.com'

In [9]:
soup.h1.parent.name   # what is the <h1> sitting inside?

'body'

## 3 · `find` vs `find_all`

The two searching workhorses:

- `find(...)` → the **first** match, or `None` if nothing matches
- `find_all(...)` → a **list** of every match (possibly empty)

In [10]:
html = """
<ul>
  <li>mango</li>
  <li>banana</li>
  <li>jackfruit</li>
</ul>"""

soup = BeautifulSoup(html, "html.parser")
soup.find("li")          # first match only

<li>mango</li>

In [11]:
soup.find_all("li")      # every match, in document order

[<li>mango</li>, <li>banana</li>, <li>jackfruit</li>]

In [12]:
[li.get_text() for li in soup.find_all("li")]

['mango', 'banana', 'jackfruit']

## 4 · Searching by class and id

Real pages are a sea of `<div>`s — class names are how you tell them apart.
Note the trailing underscore in `class_`: plain `class` is a reserved word in Python.

In [13]:
html = """
<p class="lead">Opening paragraph.</p>
<p class="lead dark">Another lead paragraph, dark variant.</p>
<p>Plain paragraph.</p>
<div id="footer">The footer.</div>"""

soup = BeautifulSoup(html, "html.parser")
soup.find_all("p", class_="lead")   # matches ANY element whose class LIST contains "lead"

[<p class="lead">Opening paragraph.</p>,
 <p class="lead dark">Another lead paragraph, dark variant.</p>]

In [14]:
soup.find("div", id="footer")

<div id="footer">The footer.</div>

### ⚠️ The multi-class trap

`class_="lead"` checks the class **list**. But if you pass the class as a bare second argument
with a **space** in it, BeautifulSoup compares the attribute **string verbatim** — so the same
classes in a different order silently match nothing. This exact bug ships in the course's
listings scraper.

In [15]:
snippet_a = '<p class="lead dark">A</p>'
snippet_b = '<p class="dark lead">B</p>'   # same two classes, opposite order

print(BeautifulSoup(snippet_a, "html.parser").find("p", "lead dark"))   # works...
print(BeautifulSoup(snippet_b, "html.parser").find("p", "lead dark"))   # ...silently None!

<p class="lead dark">A</p>
None


In [16]:
# the robust way: one class at a time, via class_=
print(BeautifulSoup(snippet_a, "html.parser").find("p", class_="lead"))
print(BeautifulSoup(snippet_b, "html.parser").find("p", class_="lead"))

<p class="lead dark">A</p>
<p class="dark lead">B</p>


## 5 · Attributes are a dictionary

Read them with `tag['name']` (raises `KeyError` if absent) or `tag.get('name')` (returns `None`).
`data-*` attributes — values a page carries for its own JavaScript — are read the same way, and
they are often the cleanest data on the page.

In [17]:
html = '<button class="cta" data-id="1002355771" data-city="Mumbai">Contact Owner</button>'

btn = BeautifulSoup(html, "html.parser").find("button")
btn["data-id"], btn["data-city"]

('1002355771', 'Mumbai')

In [18]:
print(btn.get("data-missing"))        # absent attribute: None, not a crash

None


In [19]:
btn.attrs                             # everything at once

{'class': ['cta'], 'data-id': '1002355771', 'data-city': 'Mumbai'}

## 6 · Getting text out

Four tools, four behaviours — the differences only show up on **mixed content**
(text and tags in the same element):

In [20]:
html = "<li>Config <strong>1 RK</strong></li>"
li = BeautifulSoup(html, "html.parser").find("li")

print(li.string)                    # None! .string only works when there is exactly one text child
print(li.get_text())                # all the text, glued together
print(li.get_text(strip=True))      # trimmed
print(list(li.stripped_strings))    # each text fragment separately
print(li.find(string=True))         # just the FIRST text node - the "label"
print(li.strong.get_text())         # just the <strong> - the "value"

None
Config 1 RK
Config1 RK
['Config', '1 RK']
Config 
1 RK


## 7 · Scope your searches

`soup.find(...)` searches the **whole page**. Search from a subtree instead — find the container
first, then search *inside it* — and you can never accidentally pick up a value from the
neighbouring card. This one habit prevents the classic scraping bug of row *n* wearing row
*n+1*'s data.

In [21]:
html = """
<div class="card">
  <h2>Flat in Adyar</h2>   <span class="price">80 Lakhs</span>
</div>
<div class="card">
  <h2>Flat in Velachery</h2> <span class="price">65 Lakhs</span>
</div>"""

soup = BeautifulSoup(html, "html.parser")
soup.find("span", class_="price")            # whole-page search: whose price is this?

<span class="price">80 Lakhs</span>

In [22]:
for card in soup.find_all("div", class_="card"):
    title = card.find("h2").get_text()               # searches INSIDE this card only
    price = card.find("span", class_="price").get_text()
    print(title, "->", price)

Flat in Adyar -> 80 Lakhs
Flat in Velachery -> 65 Lakhs


## 8 · CSS selectors: `select` and `select_one`

Everything above can also be written as a CSS selector — the same strings you saw in the
selector playground, and the same strings that work in Selenium and lxml. `select` always
returns a list; `select_one` returns the first match or `None`.

In [23]:
soup.select("div.card > h2")                  # child combinator

[<h2>Flat in Adyar</h2>, <h2>Flat in Velachery</h2>]

In [24]:
soup.select_one(".card .price").get_text()    # descendant, first match

'80 Lakhs'

In [25]:
soup.select("div.card:nth-of-type(2) h2")     # positional

[<h2>Flat in Velachery</h2>]

## 9 · When the thing isn't there

`find` returns `None` — and `None` has no `.get_text()`. On a real page some cards *will* be
missing a field, so an unguarded chain crashes on card 7 of 300 (or worse: on a different page
next week). Guard every `find` whose element is not guaranteed.

In [26]:
card = BeautifulSoup('<div class="card"><h2>Flat in Adyar</h2></div>', "html.parser")

try:
    card.find("span", class_="price").get_text()
except AttributeError as e:
    print("crash:", e)

crash: 'NoneType' object has no attribute 'get_text'


In [27]:
# the guard pattern
price_tag = card.find("span", class_="price")
price = price_tag.get_text(strip=True) if price_tag else None
print(price)

None


## 10 · Putting it together: cards → DataFrame

A miniature listings page, and the full pipeline: find the repeating container, extract one
dict per card, hand the list to pandas.

In [28]:
import pandas as pd

html = """
<div class="listing">
  <h2>Flat for Resale in Gaikwad Nagar</h2>
  <span class="loc">Gaikwad Nagar, Mumbai</span>
  <strong class="price">22 Lakhs</strong>
  <button data-id="1002355771">Contact</button>
</div>
<div class="listing">
  <h2>Flat for Sale in Malad West</h2>
  <span class="loc">Malad West, Mumbai</span>
  <strong class="price">50 Lakhs</strong>
  <button data-id="1002355263">Contact</button>
</div>
<div class="listing">
  <h2>Plot for Sale in Andheri</h2>
  <span class="loc">Andheri, Mumbai</span>
  <strong class="price">360 Crores</strong>
  <button data-id="1002347847">Contact</button>
</div>"""

soup = BeautifulSoup(html, "html.parser")

rows = []
for card in soup.find_all("div", class_="listing"):
    rows.append({
        "title": card.find("h2").get_text(strip=True),
        "locality": card.find("span", class_="loc").get_text(strip=True),
        "price": card.find("strong", class_="price").get_text(strip=True),
        "ad_id": card.find("button")["data-id"],
    })

pd.DataFrame(rows)

,title,locality,price,ad_id
0,Flat for Resale in Gaikwad Nagar,"Gaikwad Nagar, Mumbai",22 Lakhs,1002355771
1,Flat for Sale in Malad West,"Malad West, Mumbai",50 Lakhs,1002355263
2,Plot for Sale in Andheri,"Andheri, Mumbai",360 Crores,1002347847


## 11 · The real thing

The same pipeline on the **actual saved listings page** from the lecture — 400 KB of production
HTML. The page is hosted with the course material, so this cell works the same on Colab and on
your laptop; once downloaded, re-runs use the local copy.

In [29]:
import os, requests

if not os.path.exists("property_listings.html"):
    url = "https://raw.githubusercontent.com/iitm-da/da2402/master/data%20collection/data/property_listings.html"
    with open("property_listings.html", "w", encoding="utf-8") as f:
        f.write(requests.get(url).text)

page = BeautifulSoup(open("property_listings.html", encoding="utf-8").read(), "html.parser")
cards = page.find_all("div", class_="listing-card")
len(cards)

10

In [30]:
rows = []
for card in cards:
    btn = card.find("div", class_="button")
    rows.append({
        "title": card.find("h2").get_text(strip=True),
        "locality": card.find("span", class_="sk-caption-text").get_text(strip=True),
        "price": card.find("strong", class_="rupee").get_text(strip=True),
        "ad_id": btn["data-contentid"] if btn else None,      # sponsored cards differ - guard!
        "seller": card.find("div", class_="posted").strong
                      .get_text(strip=True).removeprefix("by "),
    })

pd.DataFrame(rows)

,title,locality,price,ad_id,seller
0,Flat for Resale in Gaikwad Nagar,"Gaikwad Nagar, Mumbai",22 Lakhs,1002355771,meghna
1,Flat for Resale in Malad West,"Malad West, Mumbai",22 Lakhs,1002355810,meghna
2,5 Acres Plots & Land for Sale in Andheri,"Andheri, Mumbai",360 Crores,1002347847,Anil
3,Flat for Sale in Malad West,"Malad West, Mumbai",50 Lakhs,1002355263,JASH
4,Flat for Sale in Khardi,"Khardi, Mumbai",1 Crore,1002355301,Suresh
5,Flat for Sale in New Panvel East,"New Panvel East, Mumbai",1.24 Crore,1002355314,Scarlet builders
6,High Rise Apartment for Sale in Goregaon West,"Goregaon West, Mumbai",1.25 Crore,1002355454,Gurtej Oberoi
7,Flat for Resale in Abdul Rehman Street,"Abdul Rehman Street, Mumbai",1.68 Crore,1002346772,Nik
8,Flat for Sale in Malad East,"Malad East, Mumbai",2.25 Crores,1002355066,nayan
9,Flat for Sale in Manpada,"Manpada, Mumbai",3.30 Crores,1002355396,Swapnil Navle


## Where to go from here

- The lecture deck embeds two interactive demos: **how a page like this gets built** (slide 2)
  and **BeautifulSoup on one listing card** — the card-by-card version of section 11.
- Golden rules: *scope every search to its container* (§7), *guard every `find` that can return
  `None`* (§9), and *parse a saved copy offline* — fetch a site once and politely.